<a href="https://colab.research.google.com/github/babbrian/vocabuddy-group-10/blob/main/VocaBuddy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Hello Google Colab")

Hello Google Colab


In [2]:
"""Minimal English / Chinese flashcards for Google Colab.

Paste this entire file into ONE Colab code cell and run it.
Or upload word_cards.py using Colab's Files sidebar, then run:
    %run /content/word_cards.py
Use %run, not !python: the buttons need the notebook's Python kernel.

Requires ipywidgets (normally available in Colab). If missing, run:
    %pip install ipywidgets

Cards autosave to cards.json in the current working directory.
Colab files are temporary: download a backup before ending your session.
Import accepts backups from this app and the original desktop version.
"""

import json
import random  # [新增] 隨機選取功能需要
from html import escape
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display


DATA_FILE = Path("cards.json").resolve()  # No __file__: works in notebook cells.


def parse_cards(text):
    """Validate a deck before accepting it from disk or an upload."""
    cards = json.loads(text)
    if not isinstance(cards, list) or any(
        not isinstance(card, dict)
        or any(
            not isinstance(card.get(key), str) or not card[key].strip()
            for key in ("english", "chinese")
        )
        for card in cards
    ):
        raise ValueError("Each card needs nonempty english and chinese text.")
    return [
        {key: card[key].strip() for key in ("english", "chinese")}
        for card in cards
    ]


def load_cards(path=DATA_FILE):
    return parse_cards(path.read_text(encoding="utf-8")) if path.exists() else []


def save_cards(cards, path=DATA_FILE):
    """Replace the old deck only after the new file has been written."""
    temporary = path.with_suffix(".json.tmp")
    temporary.write_text(
        json.dumps(cards, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    temporary.replace(path)


class WordCardApp:
    def __init__(self, path=DATA_FILE):
        self.path = Path(path)
        self.cards = load_cards(self.path)
        self.index = 0
        self.show_chinese = False
        self.pending_delete = False

        self.counter = widgets.Label()
        self.card = widgets.HTML()
        self.status = widgets.Label()
        self.english = widgets.Text(description="English:", placeholder="apple")
        self.chinese = widgets.Text(description="中文:", placeholder="蘋果")
        self.previous = widgets.Button(description="Previous")
        self.flip_button = widgets.Button(description="Flip / 翻面")
        self.next_button = widgets.Button(description="Next")
        self.random_button = widgets.Button(description="Random / 隨機")  # [新增]
        self.delete_button = widgets.Button(description="Delete", button_style="danger")
        self.add_button = widgets.Button(description="Add / 新增", button_style="success")
        self.download_button = widgets.Button(description="Download cards")
        self.upload = widgets.FileUpload(accept=".json", multiple=False,
                                         description="Import cards")
        self.output = widgets.Output()

        # Callbacks change the existing widgets; no separate window is needed.
        self.previous.on_click(lambda _: self.move(-1))
        self.next_button.on_click(lambda _: self.move(1))
        self.flip_button.on_click(self.flip)
        self.random_button.on_click(self.pick_random)  # [新增]
        self.add_button.on_click(self.add_card)
        self.delete_button.on_click(self.delete_card)
        self.download_button.on_click(self.download_cards)
        self.upload.observe(self.import_cards, names="value")

        self.ui = widgets.VBox([
            widgets.HTML("<h3>English / 中文 Word Cards</h3>"),
            self.counter, self.card,
            widgets.Box(
                [self.previous, self.flip_button, self.next_button,
                 self.random_button, self.delete_button],  # [新增] random_button
                layout=widgets.Layout(display="flex", flex_flow="row wrap"),
            ),
            self.english, self.chinese, self.add_button, self.status,
            widgets.HBox([self.download_button, self.upload]),
            widgets.HTML(
                "<small>Autosaved for this session. Download a backup before leaving. "
                "Import adds cards and skips exact duplicates.</small>"
            ),
            self.output,
        ], layout=widgets.Layout(width="100%", max_width="650px"))
        self.refresh()

    def refresh(self):
        self.pending_delete = False
        self.delete_button.description = "Delete"
        for button in (self.previous, self.flip_button, self.next_button, self.delete_button):
            button.disabled = not self.cards
        # [新增] 隨機只有在 2 張以上才有意義（1 張時只會顯示同一張）
        self.random_button.disabled = len(self.cards) < 2
        if self.cards:
            self.index %= len(self.cards)
            side = "chinese" if self.show_chinese else "english"
            text = self.cards[self.index][side]
            title = "中文" if self.show_chinese else "English"
            self.counter.value = f"Card {self.index + 1} / {len(self.cards)}"
        else:
            title, text = "", "Add your first English / Chinese card below."
            self.counter.value = "0 cards"
        # Escape card text so punctuation and HTML-like text display literally.
        self.card.value = (
            '<div style="border:1px solid #aaa;border-radius:10px;padding:24px;'
            'min-height:120px;max-height:300px;overflow:auto;overflow-wrap:anywhere">'
            f'<small>{title}</small>'
            f'<div style="font-size:28px;white-space:pre-wrap">{escape(text)}</div></div>'
        )

    def commit(self, cards):
        try:
            save_cards(cards, self.path)
        except OSError as error:
            self.status.value = f"Could not save: {error}"
            return False
        self.cards = cards
        return True

    def add_card(self, _=None):
        english, chinese = self.english.value.strip(), self.chinese.value.strip()
        if not english or not chinese:
            self.status.value = "Enter both English and Chinese text."
            return
        if self.commit(self.cards + [{"english": english, "chinese": chinese}]):
            self.index = len(self.cards) - 1
            self.show_chinese = False
            self.english.value = self.chinese.value = ""
            self.status.value = "Card added and saved."
            self.refresh()

    def move(self, step):
        if self.cards:
            self.index = (self.index + step) % len(self.cards)
            self.show_chinese = False
            self.status.value = ""
            self.refresh()

    def pick_random(self, _=None):  # [新增]
        """跳到隨機一張卡，刻意避開目前這張，避免按了看起來沒反應。"""
        if len(self.cards) < 2:
            return
        self.index = random.choice(
            [i for i in range(len(self.cards)) if i != self.index]
        )
        self.show_chinese = False
        self.status.value = "Jumped to a random card."
        self.refresh()

    def flip(self, _=None):
        if self.cards:
            self.show_chinese = not self.show_chinese
            self.status.value = ""
            self.refresh()

    def delete_card(self, _=None):
        if not self.cards:
            return
        if not self.pending_delete:
            self.pending_delete = True
            self.delete_button.description = "Confirm delete"
            self.status.value = "Click Confirm delete to remove this card; Flip or Next cancels."
            return
        if self.commit(self.cards[:self.index] + self.cards[self.index + 1:]):
            self.index = min(self.index, max(0, len(self.cards) - 1))
            self.show_chinese = False
            self.status.value = "Card deleted and saved."
            self.refresh()

    def import_cards(self, change):
        uploaded = change["new"]
        if not uploaded:
            return
        try:
            # ipywidgets 7 uses a dict; ipywidgets 8 uses a tuple of files.
            item = next(iter(uploaded.values())) if isinstance(uploaded, dict) else uploaded[0]
            incoming = parse_cards(bytes(item["content"]).decode("utf-8-sig"))
            merged = list(self.cards)
            seen = {(card["english"], card["chinese"]) for card in merged}
            for card in incoming:
                key = (card["english"], card["chinese"])
                if key not in seen:
                    merged.append(card)
                    seen.add(key)
            added = len(merged) - len(self.cards)
            if self.commit(merged):
                self.show_chinese = False
                self.status.value = f"Imported {added} new card(s) and saved."
                self.refresh()
        except (ValueError, KeyError, TypeError) as error:
            self.status.value = f"Import failed: {error}"
        finally:
            self.upload.value = {} if isinstance(self.upload.value, dict) else ()

    def download_cards(self, _=None):
        try:
            from google.colab import files
            save_cards(self.cards, self.path)
            with self.output:
                self.output.clear_output(wait=True)
                files.download(str(self.path))
            self.status.value = "Download requested. Keep the JSON file to import next time."
        except Exception as error:
            self.status.value = f"Download failed: {error}"


def main():
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass  # The basic app also works in Jupyter; Download is Colab-specific.
    try:
        app = WordCardApp()
    except (OSError, ValueError) as error:
        print(f"Could not load {DATA_FILE}: {error}")
        print("The existing file was left unchanged. Fix it or rename it, then rerun.")
        return None
    display(app.ui)
    return app


if __name__ == "__main__":
    # Rerunning the cell removes the previous UI to avoid two stale decks.
    old_app = globals().get("word_card_app")
    if old_app is not None:
        old_app.ui.close()
    word_card_app = main()